In [7]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\zomato-refund-fraud-analytics\data\raw\Zomato Dataset.csv')

# Always check your data after loading
print(df.shape)
print(df.head())
print(df.columns.tolist())

(45584, 20)
       ID Delivery_person_ID  Delivery_person_Age  Delivery_person_Ratings  \
0  0xcdcd      DEHRES17DEL01                 36.0                      4.2   
1  0xd987      KOCRES16DEL01                 21.0                      4.7   
2  0x2784     PUNERES13DEL03                 23.0                      4.7   
3  0xc8b6     LUDHRES15DEL02                 34.0                      4.3   
4  0xdb64      KNPRES14DEL02                 24.0                      4.7   

   Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
0            30.327968             78.046106                   30.397968   
1            10.003064             76.307589                   10.043064   
2            18.562450             73.916619                   18.652450   
3            30.899584             75.809346                   30.919584   
4            26.463504             80.372929                   26.593504   

   Delivery_location_longitude  Order_Date Time_Orderd Time_Or

In [8]:
# Set random seed so results are reproducible
np.random.seed(42)

# Generate 10,000 unique customer IDs
num_customers = 10000
customer_ids = [f'CUST{str(i).zfill(5)}' for i in range(1, num_customers + 1)]

# Randomly assign a customer to each order
df['Customer_ID'] = np.random.choice(customer_ids, size=len(df))

# Verify
print(f"Total orders: {len(df)}")
print(f"Unique customers: {df['Customer_ID'].nunique()}")
print(df['Customer_ID'].head(10))

Total orders: 45584
Unique customers: 9895
0    CUST07271
1    CUST00861
2    CUST05391
3    CUST05192
4    CUST05735
5    CUST06266
6    CUST00467
7    CUST04427
8    CUST05579
9    CUST08323
Name: Customer_ID, dtype: str


In [9]:
# Define refund reasons
normal_reasons = ['Late Delivery', 'Wrong Item', 'Missing Item', 
                  'Poor Quality', 'Damaged Packaging']
fraud_reasons = ['Item Not Received', 'Wrong Item', 'Item Not Received',
                 'Wrong Item', 'Item Not Received']  # intentionally repetitive

# Assign risk category to each customer
customer_risk = {}
for cust in customer_ids:
    rand = np.random.random()
    if rand < 0.03:
        customer_risk[cust] = 'High'
    elif rand < 0.10:
        customer_risk[cust] = 'Medium'
    else:
        customer_risk[cust] = 'Low'

# Generate refund data for each order
refund_requested = []
refund_reason = []
refund_amount = []

for _, row in df.iterrows():
    cust = row['Customer_ID']
    risk = customer_risk[cust]
    
    if risk == 'High':
        # Fraudulent — high refund rate, repetitive reasons
        requested = np.random.random() < 0.70
        reason = np.random.choice(fraud_reasons) if requested else None
        amount = round(np.random.uniform(200, 800), 2) if requested else 0
        
    elif risk == 'Medium':
        # Grey area — moderate refund rate
        requested = np.random.random() < 0.20
        reason = np.random.choice(normal_reasons) if requested else None
        amount = round(np.random.uniform(100, 400), 2) if requested else 0
        
    else:
        # Normal — low refund rate
        requested = np.random.random() < 0.05
        reason = np.random.choice(normal_reasons) if requested else None
        amount = round(np.random.uniform(50, 300), 2) if requested else 0
    
    refund_requested.append(requested)
    refund_reason.append(reason)
    refund_amount.append(amount)

# Add to dataframe
df['Refund_Requested'] = refund_requested
df['Refund_Reason'] = refund_reason
df['Refund_Amount'] = refund_amount

# Verify
print(f"Total refund requests: {df['Refund_Requested'].sum()}")
print(f"Overall refund rate: {df['Refund_Requested'].mean():.2%}")
print(df[['Customer_ID', 'Refund_Requested', 'Refund_Reason', 'Refund_Amount']].head(10))

Total refund requests: 3596
Overall refund rate: 7.89%
  Customer_ID  Refund_Requested Refund_Reason  Refund_Amount
0   CUST07271             False           NaN           0.00
1   CUST00861             False           NaN           0.00
2   CUST05391             False           NaN           0.00
3   CUST05192             False           NaN           0.00
4   CUST05735              True  Poor Quality         286.17
5   CUST06266             False           NaN           0.00
6   CUST00467             False           NaN           0.00
7   CUST04427              True  Missing Item         301.40
8   CUST05579              True    Wrong Item         228.52
9   CUST08323             False           NaN           0.00


In [ ]:
# Save processed dataset
output_path = r'C:\Users\user\OneDrive\Desktop\zomato-refund-fraud-analytics\data\processed\zomato_with_refunds.csv'

df.to_csv(output_path, index=False)

print(f"File saved successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

File saved successfully.
Final dataset shape: (45584, 24)
